# Data pre-processing

In [ ]:
import json
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
pd.set_option("display.max_colwidth", None)


def extract_all_comments(data):
    """
    Extract ALL parent comments from the raw JSON.
    Each post returns a list of comment texts.
    """
    edges = (
        data.get("edge_media_to_parent_comment", {})
            .get("edges", [])
    )
    comments = []
    for edge in edges:
        node = edge.get("node", {})
        text = node.get("text")
        if text:
            comments.append(text)
    return comments


def flatten_json(data, parent_key="", sep="."):
    """
    Generic JSON flattener used to extract post-level & owner-level metadata.
    BUT comments are extracted separately with extract_all_comments().
    """
    items = []
    for k, v in data.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k

        if isinstance(v, dict):
            items.extend(flatten_json(v, new_key, sep).items())

        elif isinstance(v, list):
            # Only flatten first element for metadata fields (comments excluded)
            if len(v) > 0 and isinstance(v[0], dict):
                items.extend(flatten_json(v[0], new_key + "[0]", sep).items())
            else:
                items.append((new_key, str(v)))

        else:
            items.append((new_key, v))

    return dict(items)


def build_clean_dataset(info_dir: Path, sample_limit: int = None) -> pd.DataFrame:

    records = []

    for i, path in enumerate(info_dir.glob("*.info")):
        if sample_limit and i >= sample_limit:
            break

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        all_comments = extract_all_comments(data)

        flat = flatten_json(data)

        record = {
            "post_id": flat.get("id"),
            "timestamp": flat.get("taken_at_timestamp"),
            "is_video": flat.get("is_video"),
            "caption_text": flat.get("edge_media_to_caption.edges[0].node.text"),
            "like_count": flat.get("edge_media_preview_like.count"),
            "comment_count": flat.get("edge_media_to_parent_comment.count"),
            "file_name": path.name,

            "owner_id": flat.get("owner.id"),
            "owner_username": flat.get("owner.username"),
            "owner_full_name": flat.get("owner.full_name"),
            "owner_is_private": flat.get("owner.is_private"),
            "owner_is_verified": flat.get("owner.is_verified"),

            "all_comments": all_comments,
        }

        records.append(record)

    df = pd.DataFrame(records)

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")
    df["is_video"] = df["is_video"].astype(bool)
    df["owner_is_private"] = df["owner_is_private"].astype(bool)
    df["owner_is_verified"] = df["owner_is_verified"].astype(bool)

    df["like_count"] = pd.to_numeric(df["like_count"], errors="coerce")
    df["comment_count"] = pd.to_numeric(df["comment_count"], errors="coerce")

    return df



In [ ]:

if __name__ == "__main__":
    info_dir = Path("dataset/info")

    print("Building clean DataFrame from raw JSON...")
    df = build_clean_dataset(info_dir, sample_limit=2000)

    print("Done! Clean dataset created.")
    print(df.head().to_string())

    # Optional: Save
    df.to_pickle("clean_posts.pkl")
    df.to_csv("clean_posts.csv", index=False)

    print("Saved to 'clean_posts.pkl' and 'clean_posts.csv'")


In [ ]:
import json
import pandas as pd
from pathlib import Path

df = pd.read_pickle("clean_posts.pkl") 
df.head()


In [ ]:
for col in df.columns:
    print("-", col)


In [ ]:
sample = df.iloc[0]
sample_df = pd.DataFrame({'Feature': sample.index, 'Value': sample.values})
with pd.option_context('display.colheader_justify', 'left'):
    print(sample_df.to_string(index=False, justify='left'))


# translating

In [ ]:
#!pip install deep-translator tqdm

from deep_translator import GoogleTranslator
from tqdm.auto import tqdm
import pandas as pd

df = pd.read_pickle("clean_posts.pkl")

translator = GoogleTranslator(source="auto", target="en")

def translate_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return text
    try:
        return translator.translate(text)
    except:
        return text

def translate_comment_list_batch(comment_list):
    if not isinstance(comment_list, list) or len(comment_list) == 0:
        return []

    sep = " ||| "
    merged = sep.join(comment_list)

    try:
        translated = translator.translate(merged)
    except:
        return translate_comment_list_batch

    if translated is None or not isinstance(translated, str):
        return comment_list

    return translated.split(sep)


translated_captions = []
translated_comments = []

print("Batch translating captions & comments...")

for idx in tqdm(range(len(df)), desc="Translation"):
    row = df.iloc[idx]

    # 1) caption single translate
    translated_captions.append(translate_text(row["caption_text"]))

    # 2) comments batch translate
    translated_comments.append(translate_comment_list_batch(row["all_comments"]))


df["caption_text_en"] = translated_captions
df["comments_en"] = translated_comments

df.to_pickle("clean_posts_translated.pkl")
df.to_csv("clean_posts_translated.csv", index=False)

print("Saved translated files: clean_posts_translated.pkl / .csv")


# Sentiment Analysis using Twitter RoBERTa (cardiffnlp)

In [ ]:

# #!pip install transformers tqdm torch

# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import pandas as pd
# from tqdm.auto import tqdm
# import numpy as np


# df = pd.read_pickle("clean_posts_translated.pkl")


# MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

# print("Loading sentiment model...")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# device = torch.device("cpu")
# model.to(device)


# LABELS = ["negative", "neutral", "positive"]
# mapping = {"negative": -1, "neutral": 0, "positive": 1}

# def softmax(x):
#     e_x = np.exp(x - np.max(x))
#     return e_x / e_x.sum()

# def get_sentiment(text):
#     if not isinstance(text, str) or text.strip() == "":
#         return "neutral"

#     try:
#         encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
#         encoded = {k: v.to(device) for k, v in encoded.items()}

#         with torch.no_grad():
#             output = model(**encoded)

#         scores = output.logits[0].cpu().numpy()
#         probs = softmax(scores)
#         label = LABELS[np.argmax(probs)]
#         return label
#     except:
#         return "neutral"

# def analyze_comment_list(comment_list):
#     if not isinstance(comment_list, list) or len(comment_list) == 0:
#         return 0.0, {"positive": 0.0, "neutral": 1.0, "negative": 0.0}

#     sentiments = [get_sentiment(c) for c in comment_list]

#     scores = [mapping[s] for s in sentiments]
#     avg_score = float(np.mean(scores))

#     total = len(sentiments)
#     ratio = {
#         "positive": sentiments.count("positive") / total,
#         "neutral": sentiments.count("neutral") / total,
#         "negative": sentiments.count("negative") / total
#     }

#     return avg_score, ratio


# caption_sentiments = []
# comment_avg_scores = []
# comment_ratios = []

# print("Running sentiment analysis...")

# for idx in tqdm(range(len(df)), desc="Sentiment Progress"):
#     row = df.iloc[idx]

#     # caption sentiment
#     caption_sentiments.append(get_sentiment(row["caption_text_en"]))

#     # comments: average + ratio
#     avg, ratio = analyze_comment_list(row["comments_en"])
#     comment_avg_scores.append(avg)
#     comment_ratios.append(ratio)

# df["caption_sentiment"] = caption_sentiments
# df["comments_sentiment_avg"] = comment_avg_scores

# df["comment_positive_ratio"] = [r["positive"] for r in comment_ratios]
# df["comment_neutral_ratio"]  = [r["neutral"]  for r in comment_ratios]
# df["comment_negative_ratio"] = [r["negative"] for r in comment_ratios]


# df.to_pickle("clean_posts_sentiment.pkl")
# df.to_csv("clean_posts_sentiment.csv", index=False)

# print("Sentiment analysis saved → clean_posts_sentiment.pkl / .csv")
# df.head()


In [ ]:
#!pip install transformers tqdm torch

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
from tqdm.auto import tqdm
import numpy as np
import re


df = pd.read_pickle("clean_posts_translated.pkl")

print("Loaded dataset:", df.shape)


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)    
    text = re.sub(r"@\w+", "", text)   
    text = re.sub(r"#", "", text)        
    text = re.sub(r"[^a-z0-9\s.,!?]", " ", text)  
    text = re.sub(r"\s+", " ", text).strip()  
    return text


print("Cleaning captions & comments...")

df["caption_text_clean"] = df["caption_text_en"].apply(clean_text)
df["comments_clean"] = df["comments_en"].apply(
    lambda lst: [clean_text(c) for c in (lst if isinstance(lst, list) else [])]
)



MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

print("Loading sentiment model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cpu")
model.to(device)

LABELS = ["negative", "neutral", "positive"]
mapping = {"negative": -1, "neutral": 0, "positive": 1}


def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()



def get_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return "neutral"

    try:
        encoded = tokenizer(
            text, 
            return_tensors="pt", 
            truncation=True, 
            max_length=256
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            output = model(**encoded)

        scores = output.logits[0].cpu().numpy()
        probs = softmax(scores)
        label = LABELS[np.argmax(probs)]
        return label
    except:
        return "neutral"


def analyze_comment_list(comment_list):
    if not isinstance(comment_list, list) or len(comment_list) == 0:
        return 0.0, {"positive": 0.0, "neutral": 1.0, "negative": 0.0}

    sentiments = [get_sentiment(c) for c in comment_list]

    scores = [mapping[s] for s in sentiments]
    avg_score = float(np.mean(scores))

    total = len(sentiments)
    ratio = {
        "positive": sentiments.count("positive") / total,
        "neutral": sentiments.count("neutral") / total,
        "negative": sentiments.count("negative") / total,
    }

    return avg_score, ratio


caption_sentiments = []
comment_avg_scores = []
comment_ratios = []

print("Running sentiment analysis... (this may take some time)")

for idx in tqdm(range(len(df)), desc="Sentiment Progress"):
    row = df.iloc[idx]

    # caption sentiment
    caption_sentiments.append(get_sentiment(row["caption_text_clean"]))

    # comments sentiment
    avg, ratio = analyze_comment_list(row["comments_clean"])
    comment_avg_scores.append(avg)
    comment_ratios.append(ratio)



df["caption_sentiment"] = caption_sentiments
df["comments_sentiment_avg"] = comment_avg_scores

df["comment_positive_ratio"] = [r["positive"] for r in comment_ratios]
df["comment_neutral_ratio"]  = [r["neutral"]  for r in comment_ratios]
df["comment_negative_ratio"] = [r["negative"] for r in comment_ratios]



df.to_pickle("clean_posts_sentiment.pkl")
df.to_csv("clean_posts_sentiment.csv", index=False)

print("🎉 DONE! Saved:")
print(" → clean_posts_sentiment.pkl")
print(" → clean_posts_sentiment.csv")

df.head()


In [ ]:
import pandas as pd

df = pd.read_csv("clean_posts_sentiment.csv")

print("Shape:", df.shape)
df.head()


In [ ]:
print("\nColumns:")
print(df.columns.tolist())

print("\nNull Count:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)


In [ ]:
df.describe()


In [ ]:
import pandas as pd
import numpy as np
import re
import requests
from tqdm.auto import tqdm
import time



print(" Loading clean_posts_sentiment.csv ...")
df = pd.read_csv("clean_posts_sentiment.csv")
print("Loaded:", df.shape)



df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")



df["caption_text_en"] = df["caption_text_en"].fillna("")
df["owner_full_name"] = df["owner_full_name"].fillna("")

df["comments_en"] = df["comments_en"].apply(lambda x: eval(x) if isinstance(x, str) else x)

df["comment_count"] = df.apply(
    lambda row: len(row["comments_en"]) if pd.isna(row["comment_count"]) else row["comment_count"],
    axis=1
)


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)  
    text = re.sub(r"@\w+", "", text) 
    text = re.sub(r"#", "", text) 
    text = re.sub(r"[^a-z0-9\s.,!?]", " ", text) 
    text = re.sub(r"\s+", " ", text).strip()
    return text

# caption clean
df["caption_text_clean"] = df["caption_text_en"].apply(clean_text)

# comments clean
df["comments_clean"] = df["comments_en"].apply(
    lambda lst: [clean_text(c) for c in lst] if isinstance(lst, list) else []
)


def extract_hashtags(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"#\w+", text)

df["hashtags"] = df["caption_text_en"].apply(extract_hashtags)
df["num_hashtags"] = df["hashtags"].apply(len)
df["has_hashtag"] = df["num_hashtags"].apply(lambda x: 1 if x > 0 else 0)



# caption length
df["caption_length"] = df["caption_text_clean"].apply(len)

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+", flags=re.UNICODE
)
df["num_emojis"] = df["caption_text_en"].apply(lambda x: len(emoji_pattern.findall(str(x))))

# contains mention
df["contains_mention"] = df["caption_text_en"].apply(lambda x: 1 if "@" in str(x) else 0)

# contains url
df["has_url"] = df["caption_text_en"].apply(lambda x: 1 if re.search(r"http\S+|www\S+", str(x)) else 0)

# simple engagement score
df["engagement_score"] = df["like_count"] + df["comment_count"]

# like-to-comment ratio
df["like_to_comment_ratio"] = df["like_count"] / (df["comment_count"] + 1)


ad_keywords = [
    "#ad", "#sponsored", "#partner", "#promotion",
    "#werbung", "#anzeige", "#collab", "#paidpartnership",
]

def detect_ad(caption):
    if not isinstance(caption, str):
        return False
    caption = caption.lower()
    return any(kw in caption for kw in ad_keywords)

df["is_ad_text"] = df["caption_text_en"].fillna("").apply(detect_ad)


print("\nFetching followers via HTML scraping...")
HEADERS = {"User-Agent": "Mozilla/5.0"}

def get_followers_html(username):
    if not isinstance(username, str) or username.strip() == "":
        return None
    
    url = f"https://www.instagram.com/{username}/"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=5)
        if resp.status_code != 200:
            return None

        match = re.search(r'"edge_followed_by":{"count":(\d+)}', resp.text)
        if match:
            return int(match.group(1))
        return None

    except:
        return None


usernames = df["owner_username"].dropna().unique()
followers_map = {}

for u in tqdm(usernames, desc="Scraping followers"):
    followers_map[u] = get_followers_html(u)
    time.sleep(0.2)

df["followers"] = df["owner_username"].map(followers_map)
df["followers"] = df["followers"].fillna(1).replace(0, 1)



df["engagement_rate"] = (df["like_count"] + df["comment_count"]) / df["followers"]



drop_cols = [
    "caption_text",
    "all_comments",
    "owner_id",
    "owner_full_name",
    "owner_is_private",
    "file_name"
]
df = df.drop(columns=drop_cols, errors="ignore")


df.to_csv("clean_posts_featured_with_engagement.csv", index=False)
df.to_pickle("clean_posts_featured_with_engagement.pkl")

print("\n🎉 Final dataset saved → clean_posts_featured_with_engagement.csv / .pkl")
df.head()


In [ ]:
import pandas as pd

df = pd.read_csv("followers_output.csv")
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.duplicated().sum()
df[df.duplicated()]


In [ ]:
df.describe(include="all")


# merge two files

In [ ]:
import pandas as pd
import numpy as np

followers = pd.read_csv("followers_output.csv")
posts = pd.read_csv("clean_posts_featured_with_engagement.csv")

merged_df = posts.merge(
    followers,
    on="owner_username",
    how="left",
    suffixes=("", "_new")
)

if "followers_new" in merged_df.columns:
    merged_df["followers"] = merged_df["followers_new"]
    merged_df = merged_df.drop(columns=["followers_new"])

merged_df["engagement_rate"] = (merged_df["like_count"] + merged_df["comment_count"]) / merged_df["followers"]
merged_df.loc[merged_df["followers"].isna(), "engagement_rate"] = np.nan

merged_df.to_csv("merged_posts_followers.csv", index=False)


In [ ]:
merged_df.head()


In [ ]:
import pandas as pd
import numpy as np


posts = pd.read_csv("clean_posts_featured_with_engagement.csv")
followers = pd.read_csv("followers_output.csv")

followers = (
    followers.sort_values(["owner_username", "followers"], ascending=[True, False])
             .drop_duplicates("owner_username", keep="first")
)

df = posts.merge(
    followers,
    on="owner_username",
    how="left",
    suffixes=("", "_new")
)

if "followers_new" in df.columns:
    df["followers"] = df["followers_new"].combine_first(df["followers"])
    df = df.drop(columns=["followers_new"])

df["followers"] = pd.to_numeric(df["followers"], errors="coerce")


ad_keywords = [
    "#ad", "#sponsored", "#partner", "#promotion",
    "#werbung", "#anzeige", "#collab", "#paidpartnership"
]

def detect_ad_from_caption(text: str) -> bool:
    if not isinstance(text, str):
        return False
    caption_lower = text.lower()
    return any(kw in caption_lower for kw in ad_keywords)

source_col = "caption_text_en" if "caption_text_en" in df.columns else "caption_text"
df[source_col] = df[source_col].fillna("")
df["is_ad_text"] = df[source_col].apply(detect_ad_from_caption)


engagement_numerator = df["like_count"].fillna(0) + df["comment_count"].fillna(0)
df["engagement_rate"] = engagement_numerator / df["followers"].replace({0: np.nan})

df.loc[df["followers"].isna(), "engagement_rate"] = np.nan


print(df[["followers", "engagement_rate"]].describe())

print(df[["followers", "engagement_rate", "is_ad_text"]].isna().sum())

print("\nis_ad_text :")
print(df["is_ad_text"].value_counts(dropna=False))

dup_users = df["owner_username"].duplicated().sum()


df.to_csv("clean_posts_ready.csv", index=False)
df.to_pickle("clean_posts_ready.pkl")